---
## 2. Informed Data Cleaning — From 443 Columns to an Analysis-Ready Frame

This phase takes `master_frame.parquet` (21,347 households × 443 columns,
clean names applied) and reduces it to a clean, analysis-ready frame,
following **Phase 2** of the project workflow. Every drop or recoding
decision is **rule-based and printed for audit** — nothing is removed
silently, and every step feeds the Preprocessing Summary Table required
for the report.

**Cleaning pipeline, in order:**
1. Build a full column audit (missingness, cardinality, dominant value, source module)
2. Stage-1 structural drops: free-text "other/specify" fields, admin duplicates, dead/near-empty columns
3. Multi-select indicator pruning: drop near-never-selected sub-items
4. Structural missingness recoding for tenure-conditional modules (rental "k-", owned "l-")
5. Remaining missing-value treatment (median/mode/constant imputation, all logged)
6. Outlier treatment for monetary variables (Winsorizing, not deletion)
7. Consistency checks (ranges, duplicates, logical contradictions)
8. Pillar map — reference for Phase 3 grouping before modelling
9. Save `master_frame_clean.parquet` + Preprocessing Summary Table
10. Quick exploration of the cleaned frame


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL: Phase 2 setup — load the renamed master frame
# ══════════════════════════════════════════════════════════════════════════════

import re
import numpy as np
import pandas as pd

# If continuing in the same session, master_clean already exists in memory.
# Otherwise, reload from disk:
try:
    df = master_clean.copy()
except NameError:
    df = pd.read_parquet(PQ / 'master_frame.parquet')

# Recover the original survey codes so we can group columns by source module
INVERSE_MAP = {clean: orig for orig, clean in RENAME_MAP.items()}

print(f"Starting shape : {df.shape}")
print(f"Memory usage   : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")


### 2.1 Column Audit

Before dropping anything, build a single reference table for every column:
% missing, distinct-value count, dominant-value share, and the **source
module** it came from (recovered from the original KNBS variable code, e.g.
`c01_1` → module `c` = water/energy/assets, `k09` → module `k` = rental).
This module tag is what lets us apply tenure-aware rules to the rental and
owned-dwelling blocks later.

In [ ]:
# ── 2.1  Full column audit ───────────────────────────────────────────────────

def get_module(clean_name):
    orig = INVERSE_MAP.get(clean_name, clean_name)
    m = re.match(r'^([a-z]+)\d', orig)            # e.g. c01_1 -> 'c', g05__1 -> 'g'
    if m:
        return m.group(1)
    return orig.split('_')[0]                      # aggregates: cty_, dw_, lp_, wsvc_, ...

MODULE_LABELS = {
    'a': 'admin_geo',           'b': 'individual_raw',
    'c': 'water_energy_assets', 'd': 'dwelling_hh_module',
    'e': 'environment_hazards', 'g': 'income_expenditure',
    'h': 'housing_perception',  'i': 'land_ownership',
    'j': 'tenure_mobility',     'k': 'rental_module',
    'l': 'owned_dwelling',
}

def dominant_share(s):
    vc = s.value_counts(normalize=True, dropna=True)
    return round(float(vc.iloc[0]) * 100, 2) if len(vc) else np.nan

audit = pd.DataFrame(index=df.columns)
audit['orig_code']    = [INVERSE_MAP.get(c, c) for c in df.columns]
audit['module']       = [get_module(c) for c in df.columns]
audit['module_label'] = audit['module'].map(MODULE_LABELS).fillna('derived_aggregate')
audit['dtype']        = df.dtypes.astype(str)
audit['pct_missing']  = (df.isna().mean() * 100).round(2)
audit['n_unique']     = df.nunique(dropna=True)
audit['dominant_pct'] = [dominant_share(df[c]) for c in df.columns]

print("Columns per module:")
print(audit['module_label'].value_counts().to_string())
print(f"\nColumns with >50% missing : {(audit['pct_missing'] > 50).sum()}")
print(f"Columns with >90% missing : {(audit['pct_missing'] > 90).sum()}")
print(f"Constant columns (1 value): {(audit['n_unique'] <= 1).sum()}")


### 2.2 Stage-1 Structural Drops

Four categories of columns carry essentially **zero analytical value** and
are removed first, *unless* they are in `CORE_KEEP` — the set of household-
level variables identified in the Variable Registry (Sec 1.3) plus all
derived aggregate blocks (dwelling, individual, land-parcel, county, NEMA,
water-services, mortgage, loan, financier). High missingness in `CORE_KEEP`
variables is treated separately in 2.4 (it is usually *structural*, not a
data-quality problem).

| Category | Why it's dropped |
|---|---|
| `*_other`, `*_text`, `*_specify` | Free-text fields, near-100% missing, not usable for stats |
| Admin duplicates (`county_code_str`, `survey_tag`, `hh_uuid`, ...) | Redundant identifiers |
| Constant columns | Zero variance — no information |
| >97% missing (outside `CORE_KEEP`) | Insufficient data to inform any pillar |


In [ ]:
# ── 2.2a  Core variables — protected from auto-drop ─────────────────────────

CORE_KEEP = {
    # geography / weights
    'hh_id', 'county_code', 'urban_rural', 'hh_weight',
    # water & sanitation
    'water_src_main', 'water_collect_method', 'water_treated', 'water_dist_mins',
    'water_src_secondary', 'toilet_type', 'has_handwash_facility',
    'handwash_materials', 'spend_water_kes',
    # energy
    'lighting_src', 'electricity_hrs_day', 'electricity_conn_type',
    'cooking_fuel', 'cooking_stove_type', 'spend_electricity_kes', 'spend_energy_other_kes',
    # assets
    'owns_radio', 'owns_mobile', 'owns_tv', 'owns_computer',
    'owns_motorcycle', 'owns_vehicle', 'owns_fridge', 'has_internet',
    # income & expenditure
    'spend_food_kes', 'spend_clothing_kes', 'spend_education_kes', 'spend_health_kes',
    'spend_transport_kes', 'spend_comms_kes', 'spend_recreation_kes', 'spend_housing_kes',
    'spend_energy_kes', 'spend_other_kes', 'spend_remittances_kes',
    'pays_rent', 'rent_monthly_kes', 'tenure_type', 'owns_other_property',
    # housing problems (g05) & perception (h01-h11)
    'prob_overcrowding', 'prob_poor_water', 'prob_poor_sanitation', 'prob_poor_drainage',
    'prob_poor_road', 'prob_insecurity', 'prob_high_cost', 'prob_poor_structure',
    'perc_structure', 'perc_roof', 'perc_walls', 'perc_floor', 'perc_ventilation',
    'perc_lighting', 'perc_water', 'perc_sanitation', 'perc_waste', 'perc_security', 'perc_overall',
    # tenure & mobility
    'is_owner_occupier', 'has_title_doc', 'housing_cost_burden', 'missed_payment',
    'eviction_risk', 'yrs_in_dwelling', 'satisfied_tenure', 'hh_tenure_type',
    'n_hh_in_building', 'tenancy_duration_cat', 'year_occupied_cat', 'building_floor_cat',
    # rental module (core)
    'has_written_lease', 'rent_actual_kes', 'lease_type', 'rent_arrears', 'psu_min_rent_kes',
    # owned dwelling module (core)
    'dwelling_yr_built', 'mortgage_repayment_kes', 'dwelling_value_kes',
    'imputed_rent_kes', 'plot_size_decimals', 'had_renovation',
    # environment & hazards
    'waste_disposal_method', 'near_waste_dump', 'flood_exposure', 'erosion_exposure', 'terrain_type',
    # land ownership
    'owns_land',
    # derived household indicators
    'util_income_ratio', 'pays_utilities', 'is_slum', 'settlement_plan_status',
    'cty_med_util_ratio', 'cty_min_utility_kes', 'cty_med_bedrooms',
}

# All derived aggregate blocks (dwelling/individual/land-parcel/county/NEMA/
# water-services/mortgage/loan/financier) are protected wholesale -- they are
# purpose-built, low-cardinality summaries from Phase 1.
CORE_KEEP |= set(audit.index[audit['module_label'] == 'derived_aggregate'])
CORE_KEEP &= set(df.columns)   # keep only columns that actually exist

print(f"Core variables protected from auto-drop: {len(CORE_KEEP)}")


In [ ]:
# ── 2.2b  Apply stage-1 structural drops ─────────────────────────────────────

# (a) Free-text "other / specify" fields
FREE_TEXT_RX = re.compile(r'(other_text|other_specify|_other\d*$|_other$|_text$|_specify$)', re.I)
free_text_cols = [c for c in df.columns if FREE_TEXT_RX.search(c) and c not in CORE_KEEP]

# (b) Admin duplicates / non-analytical identifiers
admin_drop = [c for c in [
    'survey_tag', 'sample_round', 'selected_respondent_age',
    'county_code_str', 'hh_serial', 'interview_result', 'hh_uuid'
] if c in df.columns]

if {'county_code', 'county_code_str'}.issubset(df.columns):
    agree = (df['county_code'].astype(str).str.zfill(2)
             == df['county_code_str'].astype(str).str.zfill(2)).mean()
    print(f"county_code vs county_code_str agreement: {agree:.2%}  -> dropping county_code_str")

# (c) Dead columns -- constant for every household
constant_cols = [c for c in df.columns
                 if df[c].nunique(dropna=True) <= 1 and c not in CORE_KEEP]

# (d) Near-empty columns (>97% missing), outside the protected core set
near_empty_cols = audit.index[(audit['pct_missing'] > 97) & (~audit.index.isin(CORE_KEEP))].tolist()

stage1_drop = sorted(set(free_text_cols + admin_drop + constant_cols + near_empty_cols))

print(f"Free-text / specify fields : {len(free_text_cols)}")
print(f"Admin duplicates           : {len(admin_drop)}")
print(f"Constant columns           : {len(constant_cols)}")
print(f"Near-empty (>97% missing)  : {len(near_empty_cols)}")
print(f"TOTAL stage-1 drops        : {len(stage1_drop)}")

df = df.drop(columns=stage1_drop)
print(f"\nShape after stage 1: {df.shape}")


### 2.3 Multi-Select Indicator Pruning

Several survey questions are "select all that apply" — e.g. housing
*problems* (`prob_*`), housing *programmes applied to* (`housing_program_*`),
construction *challenges* (`challenge_*`). Each option became its own 0/1
column during the rename.

We **keep substantive options even if rarely selected** — rarity itself is
meaningful (e.g. "at risk of eviction" being rare is informative). We only
**drop the residual "other" tail options** selected by fewer than 0.5% of
households, which are effectively constant and add nothing to any pillar.

In [ ]:
# ── 2.3  Detect and prune sparse multi-select sub-items ─────────────────────

multiselect_groups = {}
for c in df.columns:
    orig = INVERSE_MAP.get(c, '')
    m = re.match(r'^([a-z0-9]+)__\d+', orig)
    if m:
        multiselect_groups.setdefault(m.group(1), []).append(c)

print(f"Multi-select question groups found: {len(multiselect_groups)}")

ms_prune = []
for grp, cols in multiselect_groups.items():
    for c in cols:
        if c in CORE_KEEP:
            continue
        rate = df[c].mean(skipna=True)   # selection rate for 0/1 columns
        if pd.notna(rate) and rate < 0.005:
            ms_prune.append((grp, c, round(rate * 100, 3)))

ms_prune_df = pd.DataFrame(ms_prune, columns=['group', 'column', 'selected_pct'])
print(f"Multi-select sub-items selected by <0.5% of households (dropping): {len(ms_prune_df)}")
ms_prune_df.sort_values('selected_pct').head(20)


In [ ]:
df = df.drop(columns=ms_prune_df['column'].tolist())
print(f"Shape after multi-select pruning: {df.shape}")


### 2.4 Structural Missingness — Tenure-Conditional Modules

Rental ("k_") variables are only asked of **tenants**; owned-dwelling ("l_")
variables are only asked of **owner-occupiers**. Their NaNs are therefore
**not random** — they encode "not applicable", and naively imputing them
would create false signal. We:

1. Create explicit applicability flags `is_renter` / `is_owner`
2. Recode categorical/ordinal/binary `k_`/`l_` NaNs to **`-1` = "Not applicable"**
3. Leave continuous KES variables (e.g. `rent_actual_kes`, `dwelling_value_kes`)
   as NaN — their applicability is fully explained by `is_renter`/`is_owner`,
   so they should be excluded (not imputed) from continuous-variable summaries
   for the *other* tenure group.

In [ ]:
# ── 2.4  Recode structural missingness for k_/l_ modules ─────────────────────

k_cols = [c for c in df.columns if get_module(c) == 'k']
l_cols = [c for c in df.columns if get_module(c) == 'l']

print(f"Rental-module columns (k_*) : {k_cols}")
print(f"Owned-dwelling columns (l_*): {l_cols}")

if 'pays_rent' in df.columns:
    df['is_renter'] = df['pays_rent'].eq(1).astype(int)
if 'tenure_type' in df.columns:
    df['is_owner'] = df['tenure_type'].eq(1).astype(int)

for c in k_cols + l_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        n_missing = df[c].isna().sum()
        if df[c].dropna().nunique() <= 10:        # categorical / ordinal / binary
            df[c] = df[c].fillna(-1)               # -1 = "Not applicable"
        # else: continuous KES variable -> leave NaN, covered by is_renter/is_owner
        print(f"{c:<28} structural-missing: {n_missing:>6} ({n_missing/len(df):.1%})")


### 2.5 Remaining Missing-Value Treatment

After stages 1–4, remaining missingness should be modest and largely random.
Treatment by type, all logged for the Preprocessing Summary Table:

- **Numeric, ≤10 distinct values** (binary/ordinal/categorical-coded) → impute **mode**
- **Numeric, continuous** → impute **median** (robust to skew)
- **Object/string categorical** → impute constant `"Unknown"`

In [ ]:
# ── 2.5  Impute remaining (non-structural) missing values ───────────────────

remaining_missing = (df.isna().mean() * 100).round(2)
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)
print(f"Columns with remaining missing values: {len(remaining_missing)}")
print(remaining_missing.head(20))

impute_log = []
for c in remaining_missing.index:
    pct = remaining_missing[c]
    if pd.api.types.is_numeric_dtype(df[c]):
        if df[c].dropna().nunique() <= 10:
            fill = df[c].mode(dropna=True)
            fill_val = fill.iloc[0] if len(fill) else -1
            df[c] = df[c].fillna(fill_val)
            impute_log.append((c, 'mode', fill_val, pct))
        else:
            fill_val = df[c].median()
            df[c] = df[c].fillna(fill_val)
            impute_log.append((c, 'median', round(fill_val, 2), pct))
    else:
        fill_val = 'Unknown'
        df[c] = df[c].fillna(fill_val)
        impute_log.append((c, 'constant', fill_val, pct))

impute_log_df = pd.DataFrame(impute_log, columns=['column', 'method', 'fill_value', 'pct_missing_before'])
impute_log_df


### 2.6 Outlier Treatment — Monetary Variables

KES-denominated variables (expenditure, rent, dwelling value, mortgage
repayment, savings, loans, etc.) are heavily right-skewed, as is typical of
household survey data. We:

1. Clip impossible **negative values** to 0
2. **Winsorize** (cap, not remove) at the 1st/99th percentile, computed only
   over **non-zero** values, so legitimate "no expenditure" records are
   untouched

In [ ]:
# ── 2.6  Winsorize monetary (KES) columns ────────────────────────────────────

money_cols = [c for c in df.columns if c.endswith('_kes')]
print(f"Monetary columns: {len(money_cols)}")

outlier_log = []
for c in money_cols:
    n_neg = (df[c] < 0).sum()
    df[c] = df[c].clip(lower=0)

    nonzero = df.loc[df[c] > 0, c]
    if len(nonzero) < 30:
        continue
    lo, hi = nonzero.quantile([0.01, 0.99])
    n_capped_hi = (df[c] > hi).sum()
    n_capped_lo = ((df[c] < lo) & (df[c] > 0)).sum()
    df[c] = df[c].clip(lower=lo if lo > 0 else df[c].min(), upper=hi)
    outlier_log.append((c, n_neg, n_capped_lo, n_capped_hi, round(float(lo), 2), round(float(hi), 2)))

outlier_log_df = pd.DataFrame(outlier_log, columns=[
    'column', 'negatives_clipped', 'capped_low', 'capped_high', 'p1', 'p99'
])
outlier_log_df


### 2.7 Consistency Checks

In [ ]:
# ── 2.7  Consistency checks ───────────────────────────────────────────────────

if 'hh_id' in df.columns:
    print(f"Duplicate hh_id rows: {df['hh_id'].duplicated().sum()}")

if 'county_code' in df.columns:
    bad_county = ~df['county_code'].between(1, 47)
    print(f"county_code outside 1-47: {bad_county.sum()}")

if 'dwelling_yr_built' in df.columns:
    bad_year = ~df['dwelling_yr_built'].between(1900, 2026) & df['dwelling_yr_built'].notna()
    print(f"dwelling_yr_built outside 1900-2026: {bad_year.sum()} -> set to NaN")
    df.loc[bad_year, 'dwelling_yr_built'] = np.nan

if 'hh_size' in df.columns:
    bad_size = ~df['hh_size'].between(1, 30)
    print(f"hh_size outside 1-30: {bad_size.sum()}")

if {'pays_rent', 'rent_actual_kes'}.issubset(df.columns):
    contradiction = (df['pays_rent'].ne(1)) & (df['rent_actual_kes'] > 0)
    print(f"Non-renters with rent_actual_kes > 0: {contradiction.sum()} (flagged, not altered)")

print(f"\nFinal cleaned shape: {df.shape}")


### 2.8 Pillar Map — Reference for Phase 3

This is **not applied now** — it documents how the cleaned columns map onto
the analytical pillars for the next stage of feature engineering /
modelling. The sanity check below confirms every pillar variable survived
cleaning (or flags any that didn't, so the map can be adjusted).

In [ ]:
# ── 2.8  Pillar map (reference only) ─────────────────────────────────────────

PILLAR_MAP = {
    'Water & Sanitation': [
        'water_src_main', 'water_collect_method', 'water_treated', 'water_dist_mins',
        'water_src_secondary', 'toilet_type', 'has_handwash_facility',
        'handwash_materials', 'spend_water_kes', 'prob_poor_water', 'prob_poor_sanitation',
        'perc_water', 'perc_sanitation', 'wsvc_water_conns', 'wsvc_sewer_conns',
        'wsvc_tariff', 'wsvc_quality', 'wsvc_n_providers',
    ],
    'Energy & Cooking': [
        'lighting_src', 'electricity_hrs_day', 'electricity_conn_type', 'cooking_fuel',
        'cooking_stove_type', 'spend_electricity_kes', 'spend_energy_other_kes',
        'spend_energy_kes', 'perc_lighting', 'cty_min_utility_kes', 'util_income_ratio',
    ],
    'Housing Structure & Quality': [
        'dw_type', 'dw_wall_mat', 'dw_roof_mat', 'dw_floor_mat', 'dw_rooms',
        'dw_area_m2', 'dw_bedrooms', 'dw_approved', 'dw_has_planning',
        'perc_structure', 'perc_roof', 'perc_walls', 'perc_floor',
        'perc_ventilation', 'prob_poor_structure', 'prob_overcrowding',
    ],
    'Tenure, Land & Security': [
        'hh_tenure_type', 'tenure_type', 'is_owner_occupier', 'has_title_doc',
        'owns_land', 'lp_n_parcels', 'lp_tenure_type', 'lp_has_title', 'lp_land_use',
        'lp_has_dispute', 'lp_is_registered', 'lp_used_as_collateral',
        'missed_payment', 'eviction_risk', 'satisfied_tenure', 'yrs_in_dwelling',
        'is_renter', 'is_owner',
    ],
    'Affordability & Finance': [
        'spend_food_kes', 'spend_clothing_kes', 'spend_education_kes', 'spend_health_kes',
        'spend_transport_kes', 'spend_comms_kes', 'spend_recreation_kes', 'spend_housing_kes',
        'spend_other_kes', 'spend_remittances_kes', 'rent_actual_kes', 'rent_monthly_kes',
        'psu_min_rent_kes', 'mortgage_repayment_kes', 'dwelling_value_kes', 'imputed_rent_kes',
        'housing_cost_burden', 'mort_rate', 'mort_ltv', 'mort_term_yrs',
        'loan_avg_kes', 'loan_outstanding_kes', 'fin_portfolio_kes',
    ],
    'Environment & Hazards': [
        'waste_disposal_method', 'near_waste_dump', 'flood_exposure', 'erosion_exposure',
        'terrain_type', 'dw_in_hazard_zone', 'prob_poor_drainage', 'perc_waste',
    ],
    'Demographics & Wellbeing': [
        'hh_size', 'n_female', 'n_children', 'n_elderly', 'n_working_age',
        'hh_head_sex', 'has_disability', 'max_edu_isced', 'mean_age', 'dependency_ratio',
        'prob_insecurity', 'perc_security', 'perc_overall', 'is_slum',
    ],
    'Geography & Weights': [
        'hh_id', 'county_code', 'urban_rural', 'hh_weight',
        'cty_housing_stock', 'cty_housing_backlog', 'cty_planning_staff',
        'cty_has_housing_policy', 'cty_approval_system', 'settlement_plan_status',
    ],
}

# Sanity check: which pillar variables survived cleaning?
for pillar, cols in PILLAR_MAP.items():
    present = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    flag = f"  -- MISSING: {missing}" if missing else ""
    print(f"{pillar:<28} {len(present)}/{len(cols)} present{flag}")


### 2.9 Save Cleaned Frame + Preprocessing Summary Table

In [ ]:
# ── 2.9  Save master_frame_clean.parquet + summary table ─────────────────────

master_frame_clean = df.copy()
master_frame_clean.to_parquet(PQ / 'master_frame_clean.parquet')

print(f"Original columns : 443")
print(f"Cleaned columns  : {master_frame_clean.shape[1]}")
print(f"Rows             : {master_frame_clean.shape[0]}")
print("\n✓  master_frame_clean.parquet saved.")

# Preprocessing summary table -- for the report (Phase 2 output)
summary_rows = [
    ('Free-text "other/specify" fields', len(free_text_cols),
     'Dropped — unusable for quantitative analysis, near-100% missing'),
    ('Admin duplicate identifiers', len(admin_drop),
     'Dropped — redundant with hh_id / county_code'),
    ('Constant columns', len(constant_cols),
     'Dropped — zero variance, no analytical value'),
    ('Near-empty columns (>97% missing)', len(near_empty_cols),
     'Dropped — insufficient data to inform any pillar'),
    ('Sparse multi-select sub-items (<0.5% selected)', len(ms_prune_df),
     'Dropped — effectively constant'),
    ('Rental/owned module structural NaNs', len(k_cols) + len(l_cols),
     'Recoded to -1 ("Not applicable") + applicability flags (is_renter, is_owner)'),
    ('Remaining missing values', len(impute_log_df),
     'Imputed (median / mode / "Unknown") — see impute_log_df'),
    ('Monetary columns Winsorized', len(outlier_log_df),
     'Negatives clipped to 0; capped at 1st/99th percentile of non-zero values'),
]

preprocessing_summary = pd.DataFrame(summary_rows, columns=['Issue', 'Count', 'Treatment'])
preprocessing_summary


### 2.10 Quick Exploration of the Cleaned Frame

A first look at the cleaned dataset's shape, types, and distributions —
sets up Phase 3 (EDA) and the Phase 3.3 normality gate.

In [ ]:
# ── 2.10a  Cleaned-frame overview ─────────────────────────────────────────────

num_cols = master_frame_clean.select_dtypes(include=np.number).columns
cat_cols = master_frame_clean.select_dtypes(exclude=np.number).columns

print(f"Final shape                : {master_frame_clean.shape}")
print(f"Numeric columns            : {len(num_cols)}")
print(f"Categorical/object columns : {len(cat_cols)}")

desc = master_frame_clean[num_cols].agg([
    'mean', 'median', 'std',
    lambda x: x.max() - x.min(),
    lambda x: x.quantile(0.75) - x.quantile(0.25)
]).T
desc.columns = ['Mean', 'Median', 'Std Dev', 'Range', 'IQR']
desc.head(20)


In [ ]:
# ── 2.10b  Any remaining (structural-by-design) missingness ─────────────────

remaining_na = master_frame_clean.isna().mean().sort_values(ascending=False)
remaining_na = remaining_na[remaining_na > 0]
print(f"Columns still containing NaN (structural -- continuous k_/l_ vars): {len(remaining_na)}")
remaining_na
